In [0]:
%sql
SELECT current_catalog(), current_schema();
SHOW CATALOGS;
DESCRIBE CATALOG EXTENDED sentinel_dev;

In [0]:
# ============================================================
# SENTINEL COMMERCE
# Notebook: 03_silver_orders_quality
#
# Layer   : Silver
# Purpose : Type enforcement, validation and quarantine
#
# Development implementation:
#   PySpark DataFrame API
#
# Production implementation:
#   Lakeflow Spark Declarative Pipelines + Expectations
# ============================================================


# Silver Architecture
        #                BRONZE
        #                   │
        #                   ▼
        #             TRY_CAST
        #                   │
        #                   ▼
        #             DQ RULES
        #                   │
        #        ┌──────────┴──────────┐
        #        │                     │
        #      VALID                INVALID
        #        │                     │
        #        ▼                     ▼
        #  DEDUPLICATION           QUARANTINE
        #        │
        #        ▼
        #      SILVER

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_TABLE = "sentinel_dev.bronze.orders_raw"

SILVER_TABLE = "sentinel_dev.silver.orders"

QUARANTINE_TABLE = "sentinel_dev.silver.orders_quarantine"

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze records: {bronze_df.count():,}")

bronze_df.printSchema()

In [0]:
VALID_ORDER_STATUSES = [
    "PLACED",
    "CONFIRMED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED"
]

VALID_PAYMENT_METHODS = [
    "UPI",
    "CREDIT_CARD",
    "DEBIT_CARD",
    "NET_BANKING",
    "COD"
]

In [0]:
typed_df = (
    bronze_df

    .withColumn(
        "quantity_clean",
        F.expr("try_cast(quantity AS INT)")
    )

    .withColumn(
        "unit_price_clean",
        F.expr("try_cast(unit_price AS DECIMAL(18,2))")
    )

    .withColumn(
        "total_amount_clean",
        F.expr("try_cast(total_amount AS DECIMAL(18,2))")
    )

    .withColumn(
        "order_timestamp_clean",
        F.expr("try_cast(order_timestamp AS TIMESTAMP)")
    )
)

In [0]:
display(
    typed_df
        .select(
            "order_id",
            "quantity",
            "quantity_clean",
            "unit_price",
            "unit_price_clean",
            "total_amount",
            "total_amount_clean",
            "order_timestamp",
            "order_timestamp_clean"
        )
        .limit(20)
)

In [0]:
DQ_RULES = {
    "ORDER_ID_REQUIRED":
        "order_id IS NOT NULL",

    "CUSTOMER_ID_REQUIRED":
        "customer_id IS NOT NULL",

    "PRODUCT_ID_REQUIRED":
        "product_id IS NOT NULL",

    "VALID_QUANTITY":
        "quantity_clean IS NOT NULL AND quantity_clean > 0",

    "VALID_UNIT_PRICE":
        "unit_price_clean IS NOT NULL AND unit_price_clean >= 0",

    "VALID_TOTAL_AMOUNT":
        "total_amount_clean IS NOT NULL AND total_amount_clean >= 0",

    "VALID_ORDER_STATUS":
        """
        order_status IN (
            'PLACED',
            'CONFIRMED',
            'SHIPPED',
            'DELIVERED',
            'CANCELLED'
        )
        """,

    "VALID_PAYMENT_METHOD":
        """
        payment_method IN (
            'UPI',
            'CREDIT_CARD',
            'DEBIT_CARD',
            'NET_BANKING',
            'COD'
        )
        """,

    "VALID_ORDER_TIMESTAMP":
        """
        order_timestamp_clean IS NOT NULL
        AND order_timestamp_clean <= current_timestamp()
        """,

    "VALID_ORDER_AMOUNT":
        """
        abs(
            total_amount_clean -
            (quantity_clean * unit_price_clean)
        ) <= 0.01
        """
}

In [0]:
failed_rule_expressions = [

    F.when(
        ~F.expr(rule_expression),
        F.lit(rule_name)
    )

    for rule_name, rule_expression
    in DQ_RULES.items()
]

In [0]:
quality_df = (
    typed_df

    .withColumn(
        "failed_rules",
        F.array_compact(
            F.array(*failed_rule_expressions)
        )
    )

    .withColumn(
        "is_valid",
        F.size("failed_rules") == 0
    )
)

In [0]:
display(
    quality_df
        .select(
            "order_id",
            "customer_id",
            "quantity_clean",
            "unit_price_clean",
            "total_amount_clean",
            "order_status",
            "failed_rules",
            "is_valid"
        )
)

In [0]:
display(
    quality_df
        .filter(~F.col("is_valid"))
        .select(
            "order_id",
            "customer_id",
            "quantity_clean",
            "unit_price_clean",
            "total_amount_clean",
            "order_status",
            "failed_rules"
        )
)

In [0]:
valid_orders_df =(quality_df.filter(F.col("is_valid")))
quarantine_orders_df = (quality_df.filter(~F.col("is_valid")))

In [0]:
print(
    f"""
Silver candidates     : {valid_orders_df.count():,}
Quarantine candidates : {quarantine_orders_df.count():,}
"""
)

In [0]:
assert (
    valid_orders_df.count()
    + quarantine_orders_df.count()
    == quality_df.count()
)

print("Reconciliation passed.")

In [0]:
#Removing duplicates

# Why this is better than dropDuplicates()
# This:
# df.dropDuplicates(["order_id"])
# essentially says:
# Give me one.
# Below implementation says:
# Give me the latest business event, with deterministic operational tie-breakers.

duplicate_summary = (
    valid_orders_df
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.col("count").desc())
)

display(duplicate_summary)

In [0]:
dedup_window = (
    Window
        .partitionBy("order_id")
        .orderBy(
            F.col("order_timestamp_clean").desc(),
            F.col("source_file_modified_at").desc(),
            F.col("ingested_at").desc()
        )
)

In [0]:
ranked_orders_df = (
    valid_orders_df
        .withColumn(
            "record_rank",
            F.row_number().over(dedup_window)
        )
)

In [0]:
display(
    ranked_orders_df
        .filter(F.col("order_id") == "DUP-ORDER-001")
        .select(
            "order_id",
            "order_status",
            "order_timestamp_clean",
            "record_rank"
        )
        .orderBy("record_rank")
)

In [0]:
deduplicated_orders_df = (
    ranked_orders_df
        .filter(F.col("record_rank") == 1)
        .drop("record_rank")
)

In [0]:
display(
    deduplicated_orders_df
        .filter(F.col("order_id") == "DUP-ORDER-001")
        .select(
            "order_id",
            "order_status",
            "order_timestamp_clean"
        )
)

In [0]:
remaining_duplicates = (
    deduplicated_orders_df
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

assert remaining_duplicates == 0

print("Deduplication validation passed.")

In [0]:
silver_ready_df = (
    deduplicated_orders_df
        .select(
            "order_id",
            "customer_id",
            "product_id",
            "product_name",

            F.col("quantity_clean").alias("quantity"),
            F.col("unit_price_clean").alias("unit_price"),
            F.col("total_amount_clean").alias("total_amount"),

            "payment_method",
            "order_status",

            F.col("order_timestamp_clean").alias("order_timestamp"),

            "coupon_code",
            "device_type",
            "source_system",

            "source_file",
            "source_file_name",
            "source_file_modified_at",
            "ingested_at"
        )
)

In [0]:
# Conceptually:
                #     incoming record
                #           │
                #           ▼
                #  Does order_id exist?
                #      /           \
                #    NO             YES
                #    │               │
                #  INSERT       Compare timestamps
                #                    │
                #            ┌───────┴────────┐
                #            │                │
                #       source newer      source older
                #            │                │
                #          UPDATE           IGNORE

silver_exists = spark.catalog.tableExists(SILVER_TABLE)

print(f"Silver table exists: {silver_exists}")

In [0]:
if not silver_exists:

    (
        silver_ready_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(SILVER_TABLE)
    )

    print("Initial Silver table created.")

In [0]:
silver_count = spark.table(SILVER_TABLE).count()

print(f"Silver records: {silver_count:,}")

In [0]:
if spark.catalog.tableExists(SILVER_TABLE):

    silver_delta = DeltaTable.forName(
        spark,
        SILVER_TABLE
    )

    (
        silver_delta.alias("target")

        .merge(
            silver_ready_df.alias("source"),
            "target.order_id = source.order_id"
        )

        .whenMatchedUpdateAll(
            condition="""
                source.order_timestamp
                >=
                target.order_timestamp
            """
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

    print("Silver MERGE completed.")

In [0]:
display(
    spark.table(SILVER_TABLE)
        .filter(
            F.col("order_id") == "DUP-ORDER-001"
        )
        .select(
            "order_id",
            "order_status",
            "order_timestamp",
            "ingested_at"
        )
)

In [0]:
duplicate_check = (
    spark.table(SILVER_TABLE)
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(f"Duplicate order IDs in Silver: {duplicate_check}")

assert duplicate_check == 0

In [0]:
%sql
SELECT
    order_id,
    order_status,
    order_timestamp_clean
FROM sentinel_dev.silver.silver_orders_current
WHERE order_id = 'DUP-ORDER-001';